In [6]:
import os
from dotenv import load_dotenv, dotenv_values
from pathlib import Path

In [7]:
load_dotenv(Path.cwd().parent / ".env")

True

In [8]:
"""
GDELT 2.0 DOC API — minimal Python example that works
======================================================

Why the `gdeltdoc` (gdelt-doc-api) package fails right now
----------------------------------------------------------
GDELT blocks requests whose User-Agent looks non-browser. It replies with
HTTP 429 + "Please limit requests to one every 5 seconds...". The gdeltdoc
library hard-codes its own client User-Agent inside api_client._query(), so
*every* call trips this and raises gdeltdoc.errors.RateLimitError.

Two real limits to respect:
  1. User-Agent must look like a browser  -> otherwise instant 429.
  2. Max ONE request every 5 seconds       -> otherwise 429 (real rate limit).

This script talks to the DOC API directly with requests + pandas. No wrapper
needed. Only deps: `pip install requests pandas`.
"""

import time
import requests
import pandas as pd

BASE = "https://api.gdeltproject.org/api/v2/doc/doc"
# A browser-style UA is the key to not getting blocked:
HEADERS = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)"}


def _get(params, retries=4, pause=6):
    """GET with backoff for GDELT's 1-request-per-5-seconds limit."""
    for attempt in range(retries):
        r = requests.get(BASE, params=params, headers=HEADERS, timeout=30)
        if r.status_code == 429:
            time.sleep(pause)
            continue
        r.raise_for_status()
        # GDELT sometimes returns 200 + an HTML/plain error for a bad query:
        if "json" not in r.headers.get("content-type", ""):
            raise RuntimeError(f"GDELT API error: {r.text.strip()[:200]}")
        return r.json()
    raise RuntimeError("Still rate-limited after retries — wait and try again.")


def article_search(query, timespan="7d", maxrecords=25, **extra):
    """Return matching news articles as a DataFrame.

    `query` uses GDELT's syntax, e.g.:
        '"Lloyds Bank" sourcecountry:UK'
        '"Lloyds Banking Group" (mortgage OR savings)'
    `timespan` examples: "24h", "7d", "3m". Or pass startdatetime/enddatetime.
    """
    data = _get({"query": query, "mode": "artlist", "format": "json",
                 "timespan": timespan, "maxrecords": maxrecords, **extra})
    return pd.json_normalize(data.get("articles", []))


def timeline_volume(query, timespan="14d", **extra):
    """Return a daily 'volume of coverage' timeline as a DataFrame."""
    data = _get({"query": query, "mode": "timelinevol", "format": "json",
                 "timespan": timespan, **extra})
    series = data.get("timeline", [{}])[0].get("data", [])
    df = pd.json_normalize(series)
    if not df.empty:
        df["date"] = pd.to_datetime(df["date"])
    return df


def timeline_tone(query, timespan="14d", **extra):
    """Return a timeline of the AVERAGE TONE of coverage as a DataFrame.

    Tone is GDELT's sentiment score for the matching articles at each time
    bucket. It runs roughly from -100 (extremely negative) to +100 (extremely
    positive); in practice most values sit between about -10 and +10, where
    0 is neutral, negative = unfavourable coverage, positive = favourable.

    Returns a DataFrame with columns:
        date  -> the time bucket (datetime)
        tone  -> average tone of matching articles in that bucket
    """
    data = _get({"query": query, "mode": "timelinetone", "format": "json",
                 "timespan": timespan, **extra})
    series = data.get("timeline", [{}])[0].get("data", [])
    df = pd.json_normalize(series)
    if not df.empty:
        df["date"] = pd.to_datetime(df["date"])
        df = df.rename(columns={"value": "tone"})  # clearer column name
    return df



    # 1) Recent UK news mentioning Lloyds Bank

def run_main(query, sourcecountry, timespan, maxrecords):
    arts = article_search(f'{query} sourcecountry:{sourcecountry}', timespan=timespan, maxrecords=maxrecords)
    print(f"Articles found: {len(arts)}")
    if not arts.empty:
        print(arts[["seendate", "title", "domain"]].to_string(index=False))

    time.sleep(15)  # respect the 5-second limit between calls

    # 2) Coverage-volume timeline over the last 14 days
    vol = timeline_volume(f'{query}', timespan=timespan)
    print(f"\nTimeline points: {len(vol)}")
    if not vol.empty:
        print(vol.tail(5).to_string(index=False))

    time.sleep(15)  # respect the 5-second limit between calls

    # 3) Sentiment (average tone) timeline over the last 14 days
    tone = timeline_tone(f'{query}', timespan=timespan)
    print(f"\nTone points: {len(tone)}")
    if not tone.empty:
        print(tone.tail(5).to_string(index=False))
        print(f"Average tone over period: {tone['tone'].mean():.2f} "
            "(negative = unfavourable coverage)")

run_main('Japan Interest', 'UK', '7d', 10)

# ---------------------------------------------------------------------------
# OPTIONAL: if you'd rather keep using gdeltdoc's Filters() convenience,
# you can force a browser User-Agent by monkey-patching its requests call:
#
#   import gdeltdoc.api_client as ac
#   _orig = ac.requests.get
#   ac.requests.get = lambda url, **kw: _orig(
#       url, **{**kw, "headers": {"User-Agent": "Mozilla/5.0 (Macintosh)"}})
#
#   from gdeltdoc import GdeltDoc, Filters
#   gd = GdeltDoc()
#   df = gd.article_search(Filters(keyword="Lloyds Bank", country="UK",
#                                  start_date="2026-06-08", end_date="2026-06-15"))
#
# Still keep calls >=5s apart. The direct helpers above are simpler/sturdier.
# ---------------------------------------------------------------------------


Articles found: 10
        seendate                                                                                                       title            domain
20260612T083000Z Japan Airlines Pilot Claimed He Saw UFO Bigger Than an Aircraft Carrier During 45 - Minute Alaska Encounter     ibtimes.co.uk
20260614T104500Z                                    Inside abandoned £75m theme park left to rot for 20 years | World | News     express.co.uk
20260611T154500Z                           Jacob Elordi Branded  Arrogant  by Fans in Japan Amid Kendall Jenner Romance News     ibtimes.co.uk
20260614T081500Z                                                       Pares en Foco 14 a 19 / 06 : Lo que Revela el Mercado es.dailyforex.com
20260611T070000Z        From Japan With Love : London exhibition explores how NIGO reshaped fashion , music and hype culture         aol.co.uk
20260608T160000Z                                                Gold Crashes Through 200 - DMA Support , Rallies on  Peace 

RuntimeError: Still rate-limited after retries — wait and try again.